# EE-2.3: The AC power flow — two buses, Newton's method, voltage limits

## Objectives

At the end of this notebook you should be able to:

- name the three **bus types** of a power flow (slack, PQ, PV) and say which quantities are known and unknown at each
- write the **AC power flow equations** S = V · (Y·V)\* as mismatch functions
- solve them with **Newton's method** — your own solver from M1.3 — and validate the result by a power balance
- **check your power flow against pandapower**, an industry-standard tool
- trace the **P–V curve** of a load bus to its **loadability limit**, and explain why the power factor changes that limit
- use a **PV bus** to hold a voltage, and compute the reactive power it costs

This lesson joins three earlier ones: the Y-bus (EE-2.1), complex power (EE-2.2) and Newton's method for constant-power loads (M1.3). Together they are a working AC power flow — the same algorithm that PyPSA's `n.pf()` and pandapower run on networks with thousands of buses (Module 04).

In [1]:
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore", category=FutureWarning)

### Bus types: what is known, what is solved for

At every bus four quantities describe the steady state: voltage magnitude |V|, voltage angle θ, and the injected active and reactive power P and Q. Two are specified; the power flow finds the other two:

| Bus type | Typical device | Specified | Solved for |
|---|---|---|---|
| **slack** | the reference / large grid connection | \|V\|, θ = 0 | P, Q (covers the losses — EE-1.2, M1.3) |
| **PQ** | a load (or a generator with fixed output) | P, Q | \|V\|, θ |
| **PV** | a generator that controls its voltage | P, \|V\| | θ, Q |

Injections are counted as **positive into the bus**: a load of 150 MW + 50 Mvar is an injection of −150 − j50 MVA.

### The power flow equations

From EE-2.1, the currents injected at the buses are **I = Y · V**; from EE-2.2, the complex power injected at bus i is S_i = V_i · I_i\*. Together:

**S_i = V_i · (Σ_k Y_ik V_k)\***

These equations are nonlinear in the unknown voltages (products of voltages, as in M1.3), so they need Newton's method. For each PQ bus the **mismatch** is the computed injection minus the specified one, split into real (ΔP) and imaginary (ΔQ) parts; Newton drives both to zero.

The network: a 220 kV slack bus feeds a load of 150 MW + 50 Mvar over a 100 km overhead line with 0.05 + j0.4 Ω/km, so Z = 5 + j40 Ω. Units: kV, Ω, S — then V · (Y·V)\* comes out directly in MVA (kV × kA). (These are line-to-line voltages; EE-3 shows why the single-phase formulas then give the total three-phase power.)

In [2]:
V_slack = 220.0 + 0j                 # kV, angle 0
Z_line = (0.05 + 0.4j) * 100         # ohm, 100 km
y = 1 / Z_line
Y = np.array([[y, -y], [-y, y]])     # Y-bus (S)
S_load = 150 + 50j                   # MVA consumed at bus 2


def voltages(x):
    """Bus voltage phasors from the unknowns x = [theta_2 (rad), |V_2| (kV)]."""
    return np.array([V_slack, x[1] * np.exp(1j * x[0])])


def mismatch(x):
    """[dP, dQ] at bus 2 (MW, Mvar): computed injection minus specified injection."""
    V = voltages(x)
    S_inj = V[1] * np.conj(Y[1] @ V)
    d = S_inj - (-S_load)
    return np.array([d.real, d.imag])


def numerical_jacobian(F, x, h=1e-6):
    """Central differences, as in the M1.3 exercise."""
    return np.column_stack([(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h])

### Newton's method on the AC power flow

Start from a **flat start** (θ = 0, |V| = nominal), as in M1.3, and iterate J · Δx = −F(x):

In [3]:
x = np.array([0.0, 220.0])           # flat start
rows = []
for k in range(10):
    F = mismatch(x)
    rows.append({"iteration": k, "theta2_deg": np.degrees(x[0]), "V2_kV": x[1], "max_mismatch_MVA": np.abs(F).max()})
    if np.abs(F).max() < 1e-9:
        break
    x = x + np.linalg.solve(numerical_jacobian(mismatch, x), -F)
print(pd.DataFrame(rows).to_string(index=False, formatters={"max_mismatch_MVA": "{:.1e}".format,
      "theta2_deg": "{:.6f}".format, "V2_kV": "{:.6f}".format}))

 iteration theta2_deg      V2_kV max_mismatch_MVA
         0   0.000000 220.000000          1.5e+02
         1  -6.806833 207.500000          1.1e+01
         2  -7.324260 204.819363          1.9e-01
         3  -7.333065 204.770985          6.0e-05
         4  -7.333068 204.770969          6.3e-12


Four iterations, with the familiar quadratic convergence: the mismatch falls from 150 MVA — at the flat start no power flows, so the first mismatch is the whole load — to 10⁻¹¹ MVA. The load voltage settles at **204.8 kV (0.931 per unit of 220 kV)**, lagging the slack by **7.3°**.

### Validate: power balance

The slack bus must supply the load **plus** the line losses, and the losses must equal |I|²·Z (EE-2.2):

In [4]:
V = voltages(x)
S_inj = V * np.conj(Y @ V)                 # MVA injected at each bus
I_line = (V[0] - V[1]) / Z_line            # kA
losses = abs(I_line) ** 2 * Z_line         # MVA
assert np.isclose(S_inj[1], -S_load), "bus 2 injects exactly the specified load"
assert np.isclose(S_inj.sum(), losses), "slack supply = load + losses"
print(f"slack supplies {S_inj[0].real:.3f} MW + j{S_inj[0].imag:.3f} Mvar")
print(f"line losses    {losses.real:.3f} MW + j{losses.imag:.3f} Mvar")

slack supplies 152.981 MW + j73.849 Mvar
line losses    2.981 MW + j23.849 Mvar


The slack delivers 153.0 MW for a 150 MW load — 3.0 MW are lost as heat in the line — and 73.8 Mvar for a 50 Mvar load: the line itself consumes 23.8 Mvar in its reactance, eight times its active losses (X/R = 8).

### Check against pandapower

[pandapower](https://www.pandapower.org/) is an open-source power system analysis tool used in research and by network operators, and it is the tool for AC power flow in Module 04. Build the same network there and compare:

In [5]:
net = pp.create_empty_network()
b1 = pp.create_bus(net, vn_kv=220, name="slack")
b2 = pp.create_bus(net, vn_kv=220, name="load")
pp.create_ext_grid(net, b1, vm_pu=1.0, va_degree=0.0)
pp.create_line_from_parameters(net, b1, b2, length_km=100, r_ohm_per_km=0.05, x_ohm_per_km=0.4,
                               c_nf_per_km=0.0, max_i_ka=2.0)
pp.create_load(net, b2, p_mw=150, q_mvar=50)
pp.runpp(net, numba=False)

comparison = pd.DataFrame({
    "own Newton": [x[1] / 220, np.degrees(x[0]), S_inj[0].real, S_inj[0].imag, losses.real, losses.imag],
    "pandapower": [net.res_bus.vm_pu[b2], net.res_bus.va_degree[b2], net.res_ext_grid.p_mw[0],
                   net.res_ext_grid.q_mvar[0], net.res_line.pl_mw[0], net.res_line.ql_mvar[0]],
}, index=["V2 (pu)", "theta2 (deg)", "slack P (MW)", "slack Q (Mvar)", "losses P (MW)", "losses Q (Mvar)"])
assert np.allclose(comparison["own Newton"], comparison["pandapower"], rtol=1e-6)
comparison.round(6)

                 own Newton  pandapower
V2 (pu)            0.930777    0.930777
theta2 (deg)      -7.333068   -7.333068
slack P (MW)     152.981077  152.981077
slack Q (Mvar)    73.848619   73.848619
losses P (MW)      2.981077    2.981077
losses Q (Mvar)   23.848619   23.848619

Identical to six decimals. Your thirty lines of NumPy reproduce an industry tool exactly — and now you know what that tool does inside. (pandapower uses per-unit values internally, EE-3, and also models line capacitance, set to zero here to match.)

### How far can the load grow? The P–V curve

Increase the load step by step at a constant power factor and solve each time, starting Newton from the previous solution (a simple **continuation**). The load voltage traces the **P–V curve** — the AC version of M1.3's voltage collapse:

In [6]:
def solve_load(P, Q, x0, tol=1e-8, max_iter=40):
    """Newton power flow for a load P + jQ (MW, Mvar) at bus 2; None if it does not converge."""
    def F(x):
        V_ = voltages(x)
        d = V_[1] * np.conj(Y[1] @ V_) + (P + 1j * Q)
        return np.array([d.real, d.imag])
    x_ = np.array(x0, dtype=float)
    for k in range(max_iter):
        if np.abs(F(x_)).max() < tol:
            return x_, np.linalg.det(numerical_jacobian(F, x_)), k
        x_ = x_ + np.linalg.solve(numerical_jacobian(F, x_), -F(x_))
    return None, None, max_iter


def pv_curve(q_per_p, step=10.0):
    """Solve increasing loads until the power flow fails; return the solved points."""
    points, x0, P = [], [0.0, 220.0], step
    while True:
        sol, det, iters = solve_load(P, q_per_p * P, x0)
        if sol is None:
            return pd.DataFrame(points), P
        points.append({"P_MW": P, "V2_kV": sol[1], "theta2_deg": np.degrees(sol[0]), "jacobian_det": det, "iterations": iters})
        x0, P = sol, P + step


curve_95, fail_95 = pv_curve(1 / 3)          # power factor 0.949 lagging
curve_1, fail_1 = pv_curve(0.0)              # unity power factor
print(f"pf 0.949 lagging: last solution at {curve_95.P_MW.iloc[-1]:.0f} MW, no solution at {fail_95:.0f} MW")
print(f"unity pf:         last solution at {curve_1.P_MW.iloc[-1]:.0f} MW, no solution at {fail_1:.0f} MW")
curve_95.iloc[[0, 9, 19, 29, -2, -1]].round(2)

pf 0.949 lagging: last solution at 390 MW, no solution at 400 MW
unity pf:         last solution at 530 MW, no solution at 540 MW


     P_MW   V2_kV  theta2_deg  jacobian_det  iterations
0    10.0  219.16       -0.46       6477.03           3
9   100.0  210.54       -4.75       5690.17           3
19  200.0  198.05      -10.13       4556.37           3
29  300.0  179.95      -16.89       3030.18           3
37  380.0  152.50      -25.73       1130.28           4
38  390.0  145.12      -27.92        716.16           4

As the load grows, the voltage falls ever faster, the Jacobian determinant shrinks towards zero and Newton needs more iterations — the warning signs from M1.3 — until, between 390 and 400 MW, **no solution exists**: the line cannot deliver more power at this power factor, however low the voltage falls. That is the **loadability limit**, the nose of the P–V curve. Near it the voltage has already sagged to two-thirds of nominal — far below the ±10 % band operators must keep, so the practical limit comes much earlier.

The same line carries about **530 MW at unity power factor**. Reactive power demand eats transfer capacity: every Mvar the load draws has to travel through the line's reactance too. (Without resistance, the unity-power-factor limit would be V²/(2X) = 605 MW; the resistance lowers it.)

### A PV bus: holding the voltage

A sagging voltage can be held up with reactive power (EE-2.2: magnitudes follow Q). Put a generator at bus 2 that produces no active power but **controls the voltage** at 220 kV: bus 2 becomes a **PV bus**. Now |V₂| is known; only θ₂ is unknown, solved from the active power mismatch alone. The reactive power the generator must supply follows afterwards:

In [7]:
def mismatch_pv(theta):
    """Active power mismatch at bus 2 with |V2| held at 220 kV."""
    V_ = voltages([theta[0], 220.0])
    return np.array([(V_[1] * np.conj(Y[1] @ V_)).real + S_load.real])

theta = np.array([0.0])
for _ in range(10):
    if abs(mismatch_pv(theta)[0]) < 1e-9:
        break
    theta = theta + np.linalg.solve(numerical_jacobian(mismatch_pv, theta), -mismatch_pv(theta))
V_pv = voltages([theta[0], 220.0])
Q_net = (V_pv[1] * np.conj(Y[1] @ V_pv)).imag      # net reactive injection at bus 2
Q_gen = Q_net + S_load.imag                         # generator output = net injection + load

pp.create_gen(net, b2, p_mw=0.0, vm_pu=1.0)
pp.runpp(net, numba=False)
assert np.isclose(Q_gen, net.res_gen.q_mvar[0], rtol=1e-6) and np.isclose(np.degrees(theta[0]), net.res_bus.va_degree[b2], rtol=1e-6)
print(f"holding 220 kV at bus 2 needs {Q_gen:.2f} Mvar from the local generator (pandapower: {net.res_gen.q_mvar[0]:.2f})")
print(f"theta2 = {np.degrees(theta[0]):.3f} deg; slack now supplies {net.res_ext_grid.q_mvar[0]:.2f} Mvar")

holding 220 kV at bus 2 needs 78.53 Mvar from the local generator (pandapower: 78.53)
theta2 = -7.291 deg; slack now supplies -9.27 Mvar


To hold the load bus at nominal voltage, the local generator must produce about **79 Mvar** — more than the load's own 50 Mvar, because it also covers part of what the line consumes. The voltage problem is solved locally, and the slack now *absorbs* some reactive power instead of supplying it. This is why grid codes require generators — and today also wind and solar inverters — to provide reactive power for voltage control, and why a generator that hits its **reactive power limit** loses voltage control: the bus switches from PV back to PQ, and the voltage sags again (Module 04).

## Check your understanding

**Bus types and equations**

1. Count the unknowns and equations for a network with 1 slack bus, 5 PV buses and 20 PQ buses. Which buses contribute one equation, which two?
2. Why can't every bus be a PQ bus? What would be missing?

**Limits and control**

1. The operator must keep the load bus above 0.9 per unit (198 kV). Using the P–V curve, what is the largest load at power factor 0.949? How does that compare with the loadability limit?
2. In the PV-bus case, what happens to θ₂? Why does holding the voltage barely change the angle, while it changes the reactive power a lot? (EE-2.2: angles carry P, magnitudes carry Q.)